# Test the deployed tire calculator

## Goal

Open the published calculator in Chromium and verify that the Python model
loads, renders every valid tire pairing and recalculates after an input change.

In [1]:
import os
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread

from playwright.async_api import async_playwright

## Setup

The deployment workflow supplies the exact GitHub Pages URL and commit query.

In [2]:
calculator_url = os.environ["CALCULATOR_URL"]

if calculator_url == "local":
    local_site_server = ThreadingHTTPServer(
        ("127.0.0.1", 8765),
        partial(SimpleHTTPRequestHandler, directory="../docs"),
    )
    local_site_thread = Thread(
        target=local_site_server.serve_forever,
        daemon=True,
    )
    local_site_thread.start()
    calculator_url = "http://127.0.0.1:8765/"

## Test

The browser must reach the ready state without page errors. The test then
changes power from 200 W to 250 W and requires the predicted speed to
change.

In [3]:
browser_errors = []

async with async_playwright() as playwright:
    browser = await playwright.chromium.launch()
    page = await browser.new_page()
    page.on(
        "pageerror",
        lambda error: browser_errors.append(str(error)),
    )

    response = await page.goto(
        calculator_url,
        wait_until="domcontentloaded",
        timeout=120_000,
    )
    assert response is not None and response.ok
    await page.locator('body[data-ready="true"]').wait_for(
        timeout=120_000
    )
    assert "Power at pedals" in await page.locator(
        'label:has(#rider-power)'
    ).inner_text()
    assert await page.locator("#rider-power").input_value() == "200"
    assert await page.locator("#wheel-size").input_value() == "622"
    assert await page.locator("#speed-from").count() == 0
    assert await page.locator("#speed-to").count() == 0
    assert await page.get_by_text("Wind exposure").count() == 0
    model_note = await page.locator(".model-note").inner_text()
    assert "not a 105% rule" in model_note
    assert "29.8 and 31.4 mm" in model_note

    winner_card = page.locator('[data-testid="winner-card"]')
    await winner_card.wait_for()
    ranking_rows = page.locator("#ranking-table tbody tr")
    ranking_row_count = await ranking_rows.count()
    assert ranking_row_count == 99
    assert await page.locator(".table-wrap").evaluate(
        "element => element.scrollHeight > element.clientHeight"
    )

    predicted_speed_before = await page.locator(
        '[data-testid="predicted-speed"]'
    ).text_content()
    await page.locator("#rider-power").fill("250")
    await page.locator("#calculate-button").click()
    await page.wait_for_function(
        """previousSpeed => {
            const speed = document.querySelector(
                '[data-testid="predicted-speed"]'
            );
            return speed && speed.textContent !== previousSpeed;
        }""",
        arg=predicted_speed_before,
        timeout=30_000,
    )
    predicted_speed_after = await page.locator(
        '[data-testid="predicted-speed"]'
    ).text_content()

    assert predicted_speed_before != predicted_speed_after
    await page.locator("#wheel-size").select_option("584")
    await page.locator("#bike").select_option("Gravel race")
    await page.locator("#surface").select_option("Firm gravel")
    await page.locator("#calculate-button").click()
    await page.wait_for_function(
        """previousSpeed => {
            const speed = document.querySelector(
                '[data-testid="predicted-speed"]'
            );
            return speed && speed.textContent !== previousSpeed;
        }""",
        arg=predicted_speed_after,
        timeout=30_000,
    )
    gravel_speed = await page.locator(
        '[data-testid="predicted-speed"]'
    ).text_content()
    assert gravel_speed != predicted_speed_after
    assert await page.locator(
        '#runtime-status[data-state="error"]'
    ).count() == 0
    assert browser_errors == []

    winner_text = await winner_card.inner_text()
    await browser.close()

if "local_site_server" in globals():
    local_site_server.shutdown()

{
    "url": calculator_url,
    "winner": winner_text,
    "ranking_rows": ranking_row_count,
    "speed_before": predicted_speed_before,
    "speed_after": predicted_speed_after,
    "gravel_speed": gravel_speed,
    "browser_errors": browser_errors,
}

127.0.0.1 - - [31/Jul/2026 10:06:18] "GET / HTTP/1.1" 200 -


{'url': 'http://127.0.0.1:8765/',
 'winner': 'FASTEST MODELED SYSTEM\nFRONT\nSL-R 30\n\n61.5 psi · 30.7 mm mounted\n\nREAR\nSL-R 30\n\n65.4 psi · 30.7 mm mounted\n\n22.3 mph modeled steady speed at 250.0 W',
 'ranking_rows': 99,
 'speed_before': '23.8 mph',
 'speed_after': '25.8 mph',
 'gravel_speed': '22.3 mph',
 'browser_errors': []}

## Result

A completed notebook is evidence that the deployed browser runtime, Python
model, result rendering and recalculation path all worked together.